# Particle Filtering - Double Pendulum

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

from filtering import (
    simulate,
    particle_filter,
    particle_mean,
    particle_std,
)

In [ ]:
T = 10.0
dt_sim = 1./1e3  
dt_obs = 1./200.

D = 4
O = 2

q = 0.05
r = 0.3

Q = jnp.diag(jnp.array([1e-6, 1e-6, 1e-6, q**2]))
R = r**2 * jnp.eye(O)

g = 9.81
l1 = 1.0
l2 = 1.5
m1 = 2.0
m2 = 2.1
b1 = 0.25
b2 = 0.25

In [ ]:
def f(x):
    theta1 = x[..., 0]
    theta2 = x[..., 1]
    omega1 = x[..., 2]
    omega2 = x[..., 3]

    delta = theta1 - theta2

    s = jnp.sin(delta)
    c = jnp.cos(delta)

    # Elements of the mass matrix
    a = m2 * l2**2
    b = -m2 * l1 * l2 * c
    d = (m1 + m2) * l1**2

    # Determinant
    det = a * d - b**2

    # h + G + Cq
    F1 = (
        m2 * l1 * l2 * s * omega2**2
        + (m1 + m2) * g * l1 * jnp.sin(theta1)
        + b1 * omega1
    )

    F2 = (
        -m2 * l1 * l2 * s * omega1**2
        + m2 * g * l2 * jnp.sin(theta2)
        + b2 * omega2
    )

    # M^{-1} @ F
    alpha1 = (d * F1 - b * F2) / det
    alpha2 = (-b * F1 + a * F2) / det

    return jnp.stack([
        omega1,
        omega2,
        -alpha1,
        -alpha2,
    ], axis=-1)


    
def f_rk4(x, t, dt):
    k1 = f(x)
    k2 = f(x + 0.5*dt*k1)
    k3 = f(x + 0.5*dt*k2)
    k4 = f(x + dt*k3)

    return x + (dt/6.) * (k1 + 2*k2 + 2*k3 + k4)
    

def h(x):
    return x[...,0:2]

In [ ]:
key = jax.random.PRNGKey(seed=42)

x0 = jnp.array([0.0, 0.0, 0.0, 0.0])

(t_sim, x_sim), (t_obs, y_obs) = simulate(f_rk4, h, x0, T, dt_sim, dt_obs, Q, R, key)

In [ ]:
N = 50_000

particles, log_weights, ess = particle_filter(
    f_rk4,
    h,
    t_obs,
    y_obs,
    n_particles=N,
    x0_mean=jnp.zeros(D),
    x0_cov=jnp.eye(D),
    Q=Q,
    R=R,
    key=key,
    n_substeps=5,
)

In [ ]:
mean = particle_mean(particles, log_weights)
std = particle_std(particles, log_weights)

fig, axs = plt.subplots(D+1, 1, figsize=(8, 2.5*(D+1)), sharex=True)
for i in range(D):
    axs[i].plot(t_sim, x_sim[...,i], lw=0.8, alpha=0.7)
    axs[i].plot(t_obs, mean[..., i], lw=1.5, alpha=0.7)
    axs[i].fill_between(t_obs, mean[..., i] - 2 * std[..., i], mean[..., i] + 2 * std[..., i], alpha=0.3)
    axs[i].set_ylabel(f"x{i+1}")
    axs[i].grid()
for j in range(O):
    axs[j].scatter(t_obs, y_obs[...,j], marker="x", s=1.0, alpha=0.1, c="r")
axs[-1].plot(t_obs, ess)
axs[-1].axhline(y=0.5 * N, ls="--", c="black", alpha=0.3)
axs[-1].set_ylim(0, 1.1 * N)
axs[-1].set_ylabel("ESS")
axs[-1].set_xlabel("t (s)")
axs[-1].grid(True)
plt.tight_layout();